### **TIDIGITS**

In [1]:
import numpy as np
from src.gmm import GMM
from src.knn import KNN


### **Load Data**

In [2]:
sequences = []
labels = []
for digit in [1, 5, 6]:
    with np.load(f"./data/tidigits/{digit}.npz") as data:
        for key in data.files:
            sequences.append(data[key])
            labels.append(digit)

X = []
for sequence in sequences:
    parts = np.array_split(sequence, 4)
    means = [part.mean(axis=0) for part in parts]
    X.append(np.concatenate(means))
X = np.array(X)
y = np.array(labels)


### **Shuffle and Split**

In [3]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(y))
train_end = int(0.6 * len(y))
val_end = int(0.8 * len(y))

train_indices = indices[:train_end]
val_indices = indices[train_end:val_end]
test_indices = indices[val_end:]

X_train = X[train_indices]
y_train = y[train_indices]
X_val = X[val_indices]
y_val = y[val_indices]
X_test = X[test_indices]
y_test = y[test_indices]


In [4]:
classes = np.unique(y_train)
print("Class order:", classes)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Class order: [1 5 6]
Train: (810, 156)
Validation: (270, 156)
Test: (270, 156)


### **GMM**

In [5]:
models = []
priors = []
for label in classes:
    samples = X_train[y_train == label]
    model = GMM(K=3, reg=1e-3, max_iter=100, tol=1e-4)
    model.fit(samples)
    models.append(model)
    priors.append(len(samples) / len(X_train))
    print("Class", label, "EM iterations:", len(model.log_likelihoods))
    if len(model.log_likelihoods) == model.max_iter:
        print("Iteration limit reached; convergence is not guaranteed.")


Class 1 EM iterations: 2
Class 5 EM iterations: 2
Class 6 EM iterations: 2


In [6]:
val_scores = np.column_stack([model.score_samples(X_val) for model in models])
gmm_val_predictions = classes[np.argmax(val_scores + np.log(priors), axis=1)]

test_scores = np.column_stack([model.score_samples(X_test) for model in models])
gmm_test_predictions = classes[np.argmax(test_scores + np.log(priors), axis=1)]


In [7]:
# Confusion matrix
gmm_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        gmm_confusion[i, j] = np.sum(
            (y_test == actual) & (gmm_test_predictions == predicted))

print("Validation accuracy:", np.mean(gmm_val_predictions == y_val))
print("Test accuracy:", np.mean(gmm_test_predictions == y_test))
print("Test confusion matrix:\n", gmm_confusion)


Validation accuracy: 0.9037037037037037
Test accuracy: 0.9
Test confusion matrix:
 [[87  0  1]
 [25 69  0]
 [ 0  1 87]]


### **KNN**

In [8]:
knn = KNN(k=5)
knn.fit(X_train, y_train)
knn_val_predictions = knn.predict(X_val)
knn_test_predictions = knn.predict(X_test)


In [9]:
# Confusion matrix
knn_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        knn_confusion[i, j] = np.sum(
            (y_test == actual) & (knn_test_predictions == predicted))

print("Validation accuracy:", np.mean(knn_val_predictions == y_val))
print("Test accuracy:", np.mean(knn_test_predictions == y_test))
print("Test confusion matrix:\n", knn_confusion)


Validation accuracy: 0.9888888888888889
Test accuracy: 0.9777777777777777
Test confusion matrix:
 [[85  1  2]
 [ 2 91  1]
 [ 0  0 88]]
